In [1]:
from pathlib import Path
import random

from app.ingestion.pdf_loader import load_pdfs
from app.ingestion.text_cleaner import clean_text, clean_documents

from app.rag.chunker import SemantikChunker
from app.rag.embedding import EmbeddingManager
from app.rag.vector_store import VectorStore
from app.rag.retriever import Retriever
from app.generation.prompts import format_retrieved_context, build_rag_qa_prompt 

/Users/macstudio/Desktop/Development/DomainForge V1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()
RAW_DATA_DIR = BASE_DIR / "data" / "raw"

raw_data = load_pdfs(RAW_DATA_DIR)
cleaned_docs = clean_documents(raw_data)



In [3]:
chunker = SemantikChunker()
chunks = chunker.chunk_corpus(cleaned_docs)

print(f"Generated Chunk Count: {len(chunks)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11475.37it/s]


Generated Chunk Count: 2347


In [4]:
random_idx = random.randint(0, len(chunks) - 1)
sample_chunk = chunks[random_idx]

print(f"Index: {random_idx}")
print(f"Chunk ID: {sample_chunk.chunk_id}")
print(f"Source: {sample_chunk.source} (Page {sample_chunk.page})")
print(f"Text Character Count: {len(sample_chunk.text)}")
print("-" * 50)
print(sample_chunk.text)

Index: 1733
Chunk ID: NIST.AI.600-1_p63_c55
Source: NIST.AI.600-1.pdf (Page 63)
Text Character Count: 48
--------------------------------------------------
(2024) OpenAI’s GPT Is A Recruiter’s Dream Tool.


In [5]:
processed_path = Path("data/processed/chunk.jsonl")
processed_path.parent.mkdir(parents=True, exist_ok=True)

with open(processed_path, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(chunk.model_dump_json() + "\n")

print(f"The chunks have been saved:{processed_path} ({len(chunks)} Chunk)")

The chunks have been saved:data/processed/chunk.jsonl (2347 Chunk)


In [6]:
VECTORSTORE_DIR = BASE_DIR / "data" / "vectorstore"

embed_mgr = EmbeddingManager(model_name="BAAI/bge-base-en-v1.5")
vs_instance = VectorStore(
    persist_dir=VECTORSTORE_DIR,
    collection_name="domainforge_governance",
    embedding_manager=embed_mgr
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22199.76it/s]


In [7]:
vs_instance.add_chunks(chunks)

print(f"Ready! Total number of records in the database: {vs_instance.count()}")

[VECTOR STORE] 2347 chunks are being indexed to disk...


Batches: 100%|██████████| 4/4 [00:00<00:00,  5.98it/s]


[VECTOR STORE] Indexing completed and saved. Total: 2347
Ready! Total number of records in the database: 2347


In [8]:
retriever = Retriever(vector_store=vs_instance, top_k=8)

test_query = "What are the core characteristics of trustworthy AI systems according to governance frameworks?"
retrieved_contexts = retriever.retrieve(test_query)

print(f"Query: {test_query} \n ")
print(f"Retrieved Context Count: {len(retrieved_contexts)}\n")
print("=" * 60 )

for i, ctx in enumerate(retrieved_contexts, 1):
    print(f"[{i}] Similarity Score:   {ctx.score: .4f}")
    print(f"      Source Document:    {ctx.source} (Page {ctx.page})")
    print(f"      Chunk ID:           {ctx.chunk_id}")
    print(f"      Context Preview:    {ctx.text[:180]}...\n")

Query: What are the core characteristics of trustworthy AI systems according to governance frameworks? 
 
Retrieved Context Count: 8

[1] Similarity Score:    0.7847
      Source Document:    NIST.AI.100-1.pdf (Page 17)
      Chunk ID:           NIST.AI.100-1_p17_c4
      Context Preview:    Characteristics of trustworthy AI
systems include: valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and...

[2] Similarity Score:    0.7735
      Source Document:    NIST.AI.100-1.pdf (Page 27)
      Chunk ID:           NIST.AI.100-1_p27_c14
      Context Preview:    GOVERN 1.2: The characteristics of trustworthy AI are integrated into organizational policies, processes, procedures, and
practices....

[3] Similarity Score:    0.7735
      Source Document:    NIST.AI.600-1.pdf (Page 18)
      Chunk ID:           NIST.AI.600-1_p18_c0
      Context Preview:    GOVERN 1.2: The characteristics of trustworthy AI are integrated i

In [9]:
formatted_context = format_retrieved_context(retrieved_contexts)

rag_prompt = build_rag_qa_prompt(query=test_query, contexts= retrieved_contexts)

print("=== GENERATED RAG PROMPT ===\n")
print(rag_prompt)

=== GENERATED RAG PROMPT ===

You are a specialized AI Governance, Ethics, and Risk Management technical expert.
Your responses must be strictly grounded in the provided contextual references (such as NIST AI RMF, UNESCO Recommendations, and OECD AI Principles).
Do not fabricate information, extrapolate without evidence, or use outside unverified facts.

### Verified Reference Context:
[CONTEXT BLOCK START]
Source: NIST.AI.100-1.pdf
Page: 17
Chunk ID: NIST.AI.100-1_p17_c4
Content: Characteristics of trustworthy AI
systems include: valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias
managed.
[CONTEXT BLOCK END]

[CONTEXT BLOCK START]
Source: NIST.AI.100-1.pdf
Page: 27
Chunk ID: NIST.AI.100-1_p27_c14
Content: GOVERN 1.2: The characteristics of trustworthy AI are integrated into organizational policies, processes, procedures, and
practices.
[CONTEXT BLOCK END]

[CONTEXT BLOCK START]
Source

In [10]:
top_result = retrieved_contexts[0]

print("=== PROVENANCE VERIFICATION ===")
print(f"Target Chunk ID: {top_result.chunk_id}")
print(f"Source PDF:      {top_result.source}")
print(f"Original Page:   {top_result.page}")
print(f"Similarity:      {top_result.score:.4f}")
print(f"Verified Non-empty Text: {len(top_result.text) > 0}")

=== PROVENANCE VERIFICATION ===
Target Chunk ID: NIST.AI.100-1_p17_c4
Source PDF:      NIST.AI.100-1.pdf
Original Page:   17
Similarity:      0.7847
Verified Non-empty Text: True
